In [3]:
from google.colab import drive
drive.mount('/content/drive')

token = "YOUR_GITHUB_TOKEN_HERE"
repo_url = f"https://{token}@github.com/wpuzhu/EECS545-Final-Project.git"

!git clone {repo_url}
%cd EECS545-Final-Project
!git checkout yslin
!ls

!pip install -q open_clip_torch
!pip install -q scikit-learn pandas tqdm
import zipfile, os

zip_path = "/content/drive/MyDrive/chest-xrays-indiana-university.zip"
extract_path = "/content/images"
image_check_path = "/content/images/chest-xrays-indiana-university/images/valid_uid_images"

# Check images
if not os.path.exists(image_check_path) or len(os.listdir(image_check_path)) == 0:
    print("Extracting images...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Done!")
    print(f"Files extracted: {len(os.listdir(image_check_path))}")
else:
    print(f"Already extracted! Files: {len(os.listdir(image_check_path))}")

Mounted at /content/drive
Cloning into 'EECS545-Final-Project'...
remote: Enumerating objects: 238, done.
remote: Counting objects: 100% (238/238), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 238 (delta 107), reused 185 (delta 71), pack-reused 0 (from 0)
Receiving objects: 100% (238/238), 2.86 MiB | 12.88 MiB/s, done.
Resolving deltas: 100% (107/107), done.
/content/EECS545-Final-Project
Branch 'yslin' set up to track remote branch 'yslin' from 'origin'.
Switched to a new branch 'yslin'
data_preparation_padim.ipynb  indiana_padim_result	   train_padim.csv
dataset			      matched_frontal_dataset.csv  val_padim.csv
experiment_padim.ipynb	      __pycache__		   WinClip
final_clean_dataset.csv       README.md
FrequencyEncoder.py	      test_padim.csv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
Extracting images...
Done!
Files extracted: 6379


In [ ]:
import sys, os, numpy as np, pandas as pd, torch
from PIL import Image, ImageOps
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

sys.path.append("/content/EECS545-Final-Project")
sys.path.append("/content/EECS545-Final-Project/WinClip")

IMAGE_FOLDER = "/content/images/chest-xrays-indiana-university/images/valid_uid_images"
CSV_PATH = "/content/images/chest-xrays-indiana-university/final_clean_dataset.csv"
PROJ_PATH = "/content/images/chest-xrays-indiana-university/indiana_projections.csv"
device = "cuda" if torch.cuda.is_available() else "cpu"

df_text = pd.read_csv(CSV_PATH)
df_text["uid"] = df_text["uid"].astype(str)
df_text = df_text[["uid", "normal"]].dropna()
df_text["abnormal"] = 1 - df_text["normal"].astype(int)

proj_df = pd.read_csv(PROJ_PATH)
proj_df["uid"] = proj_df["uid"].astype(str)
proj_df["projection"] = proj_df["projection"].str.strip().str.lower()

uid_to_frontal = {}
for uid, group in proj_df.groupby("uid"):
    frontal = group[group["projection"] == "frontal"]
    if len(frontal) == 1:
        fname = os.path.basename(frontal.iloc[0]["filename"])
        fpath = os.path.join(IMAGE_FOLDER, fname)
        if os.path.exists(fpath):
            uid_to_frontal[uid] = fpath

df = df_text[df_text["uid"].isin(uid_to_frontal.keys())].copy()
df = df.drop_duplicates(subset=["uid"]).reset_index(drop=True)
print(f"Total: {len(df)}, Abnormal: {df['abnormal'].sum()}, Normal: {(df['abnormal']==0).sum()}")

Total: 3175, Abnormal: 2010, Normal: 1165


file check

In [ ]:
# Step 1
import os
IMAGE_FOLDER = "/content/images/chest-xrays-indiana-university/images/valid_uid_images"
files = os.listdir(IMAGE_FOLDER) if os.path.exists(IMAGE_FOLDER) else []
print(f"IMAGE_FOLDER exists: {os.path.exists(IMAGE_FOLDER)}")
print(f"Number of files in folder: {len(files)}")
if len(files) > 0:
    print(f"First 3 filenames: {files[:3]}")

# Step 2
import pandas as pd
PROJ_PATH = "/content/images/chest-xrays-indiana-university/indiana_projections.csv"
proj_df = pd.read_csv(PROJ_PATH)
proj_df["projection"] = proj_df["projection"].str.strip().str.lower()
frontal_df = proj_df[proj_df["projection"] == "frontal"]
print(f"\nTotal frontal rows: {len(frontal_df)}")
print(f"Sample filenames from CSV:")
print(frontal_df["filename"].head(5).tolist())

# Step 3
if len(files) > 0 and len(frontal_df) > 0:
    csv_sample = frontal_df["filename"].iloc[0]
    folder_sample = files[0]
    print(f"\nCSV filename example:    '{csv_sample}'")
    print(f"Folder filename example: '{folder_sample}'")

In [4]:
!git fetch origin
!git checkout origin/padim -- test_padim.csv
test_df = pd.read_csv("test_padim.csv")
print(f"Test set size: {test_df.shape}")
print(test_df.columns.tolist())
print(test_df.head(3))

NameError: name 'pd' is not defined

In [ ]:
# test set uid <--> label
test_df["uid"] = test_df["uid"].astype(str)

# picture uid
test_df_filtered = test_df[test_df["uid"].isin(uid_to_frontal.keys())].copy()
test_df_filtered = test_df_filtered.drop_duplicates(subset=["uid"]).reset_index(drop=True)
test_df_filtered["abnormal"] = test_df_filtered["label"]

print(f"Test set after filtering: {len(test_df_filtered)}")
print(f"Abnormal: {test_df_filtered['abnormal'].sum()}, Normal: {(test_df_filtered['abnormal']==0).sum()}")

Test set after filtering: 1621
Abnormal: 1389, Normal: 232


**Zero-shot CLIP (ViT-B-32)**


In [ ]:
import sys
import open_clip
import numpy as np
import torch
from PIL import Image, ImageOps
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score

sys.path.append("/content/EECS545-Final-Project")
from WinClip.WinCLIP.ad_prompts import (
    state_level_normal_prompts,
    state_level_abnormal_prompts,
    template_level_prompts,
)

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer('ViT-B-32')


# Build full prompt list using all templates x all states
category = "chest x-ray patient"
normal_texts = [t.format(n.format(category)) for t in template_level_prompts for n in state_level_normal_prompts]
abnormal_texts = [t.format(a.format(category)) for t in template_level_prompts for a in state_level_abnormal_prompts]


with torch.no_grad():
    # Normal: encode all, then mean-pool into 1 vector (normal prompts all say similar things)
    normal_tokens = tokenizer(normal_texts).to(device)
    normal_features = model.encode_text(normal_tokens)           # (104, 512)
    normal_features = normal_features / normal_features.norm(dim=-1, keepdim=True)
    normal_feat_mean = normal_features.mean(dim=0, keepdim=True) # (1, 512)
    normal_feat_mean = normal_feat_mean / normal_feat_mean.norm(dim=-1, keepdim=True)

    # Abnormal: encode all, keep individual vectors (DO NOT average)
    abnormal_tokens = tokenizer(abnormal_texts).to(device)
    abnormal_features = model.encode_text(abnormal_tokens)       # (611, 512)
    abnormal_features = abnormal_features / abnormal_features.norm(dim=-1, keepdim=True)

def resize_and_pad(img, target_size=(224, 224)):
    img = ImageOps.contain(img, target_size)
    canvas = Image.new("RGB", target_size, color=(0, 0, 0))
    canvas.paste(img, ((target_size[0]-img.size[0])//2, (target_size[1]-img.size[1])//2))
    return canvas

scores_clip, labels_clip = [], []
with torch.no_grad():
    for _, row in tqdm(test_df_filtered.iterrows(), total=len(test_df_filtered)):
        img = Image.open(uid_to_frontal[row["uid"]]).convert("RGB")
        img = resize_and_pad(img)
        img_tensor = preprocess(img).unsqueeze(0).to(device)

        feat = model.encode_image(img_tensor)            # (1, 512)
        feat = feat / feat.norm(dim=-1, keepdim=True)

        # Normal: single dot product with mean vector
        sim_normal = (feat @ normal_feat_mean.T).item()

        # Abnormal: dot product with ALL 611 vectors, take MAX
        # Logic: if the image looks like ANY known disease, it's abnormal
        sim_abnormal = (feat @ abnormal_features.T).mean().item()

        score = sim_abnormal - sim_normal
        scores_clip.append(score)
        labels_clip.append(row["abnormal"])

scores_clip = np.array(scores_clip)
labels_clip = np.array(labels_clip)
auroc = roc_auc_score(labels_clip, scores_clip)
thresholds = np.linspace(scores_clip.min(), scores_clip.max(), 100)
best_f1 = max(f1_score(labels_clip, (scores_clip > t).astype(int), average='macro') for t in thresholds)
print(f"Zero-shot CLIP (ViT-B-32) AUROC: {auroc:.4f}  Best F1: {best_f1:.4f}")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(

Zero-shot CLIP (ViT-B-32) AUROC: 0.5962  Best F1: 0.5394


**Zero-shot CLIP + FFT (ViT-B-32)**

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

from FrequencyEncoder import FrequencyEncoder
from torchvision import transforms

freq_encoder = FrequencyEncoder(in_channels=3, out_dim=512).to(device).eval()

to_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711])
])

# Reuse normal_feat_mean and abnormal_features from Cell 3
# (already built with full ad_prompts.py, 104 normal + 611 abnormal vectors)

scores_fft, labels_fft = [], []
with torch.no_grad():
    for _, row in tqdm(test_df_filtered.iterrows(), total=len(test_df_filtered)):
        img = Image.open(uid_to_frontal[row["uid"]]).convert("RGB")
        img = resize_and_pad(img)

        # CLIP spatial feature
        clip_feat = model.encode_image(preprocess(img).unsqueeze(0).to(device))
        clip_feat = clip_feat / clip_feat.norm(dim=-1, keepdim=True)

        # FFT frequency feature
        freq_feat = freq_encoder(to_tensor(img).unsqueeze(0).to(device))
        freq_feat = freq_feat / freq_feat.norm(dim=-1, keepdim=True)

        # Fuse
        fused = clip_feat + freq_feat
        fused = fused / fused.norm(dim=-1, keepdim=True)

        # Same max scoring as Cell 3
        sim_normal = (fused @ normal_feat_mean.T).item()
        sim_abnormal = (fused @ abnormal_features.T).mean().item()
        score = sim_abnormal - sim_normal

        scores_fft.append(score)
        labels_fft.append(row["abnormal"])

scores_fft = np.array(scores_fft)
labels_fft = np.array(labels_fft)
auroc_fft = roc_auc_score(labels_fft, scores_fft)
thresholds = np.linspace(scores_fft.min(), scores_fft.max(), 100)
best_f1_fft = max(f1_score(labels_fft, (scores_fft > t).astype(int), average='macro') for t in thresholds)
print("Zero-shot CLIP + FFT done!")
print(f"Zero-shot CLIP + FFT (ViT-B-32) AUROC: {auroc_fft:.4f}  Best F1: {best_f1_fft:.4f}")

100%|██████████| 1621/1621 [02:29<00:00, 10.82it/s]


Zero-shot CLIP + FFT done!
Zero-shot CLIP + FFT (ViT-B-32) AUROC: 0.6037  Best F1: 0.5442


**Zero-shot CLIP (ViT-B-16-plus-240)**

In [ ]:
import open_clip
import sys
from sklearn.metrics import roc_auc_score, f1_score

sys.path.append("/content/EECS545-Final-Project")
from WinClip.WinCLIP.ad_prompts import (
    state_level_normal_prompts,
    state_level_abnormal_prompts,
    template_level_prompts,
)

# Load ViT-B-16-plus-240
model_winclip, _, preprocess_winclip = open_clip.create_model_and_transforms(
    'ViT-B-16-plus-240', pretrained='laion400m_e32'
)
model_winclip = model_winclip.to(device).eval()
tokenizer_winclip = open_clip.get_tokenizer('ViT-B-16-plus-240')

# Build full prompt lists (same as Cell 3, but for 640-dim model)
category = "chest x-ray patient"
normal_texts_w   = [t.format(n.format(category)) for t in template_level_prompts
                                                  for n in state_level_normal_prompts]
abnormal_texts_w = [t.format(a.format(category)) for t in template_level_prompts
                                                  for a in state_level_abnormal_prompts]

with torch.no_grad():
    # Normal: mean-pool into 1 vector (640-dim)
    normal_tokens_w = tokenizer_winclip(normal_texts_w).to(device)
    normal_features_w = model_winclip.encode_text(normal_tokens_w)      # (104, 640)
    normal_features_w = normal_features_w / normal_features_w.norm(dim=-1, keepdim=True)
    normal_feat_w_mean = normal_features_w.mean(dim=0, keepdim=True)    # (1, 640)
    normal_feat_w_mean = normal_feat_w_mean / normal_feat_w_mean.norm(dim=-1, keepdim=True)

    # Abnormal: keep all individual vectors (611, 640) — DO NOT average
    abnormal_tokens_w = tokenizer_winclip(abnormal_texts_w).to(device)
    abnormal_features_w = model_winclip.encode_text(abnormal_tokens_w)  # (611, 640)
    abnormal_features_w = abnormal_features_w / abnormal_features_w.norm(dim=-1, keepdim=True)

winclip_scores, winclip_labels = [], []
with torch.no_grad():
    for _, row in tqdm(test_df_filtered.iterrows(), total=len(test_df_filtered)):
        img = Image.open(uid_to_frontal[row["uid"]]).convert("RGB")
        img = resize_and_pad(img)
        feat = model_winclip.encode_image(preprocess_winclip(img).unsqueeze(0).to(device))
        feat = feat / feat.norm(dim=-1, keepdim=True)

        # Normal: single dot product
        sim_normal = (feat @ normal_feat_w_mean.T).item()

        # Abnormal: dot product with all 611 vectors, take MAX
        sim_abnormal = (feat @ abnormal_features_w.T).mean().item()

        score = sim_abnormal - sim_normal
        winclip_scores.append(score)
        winclip_labels.append(row["abnormal"])

winclip_scores = np.array(winclip_scores)
winclip_labels = np.array(winclip_labels)
auroc_winclip = roc_auc_score(winclip_labels, winclip_scores)
thresholds = np.linspace(winclip_scores.min(), winclip_scores.max(), 100)
best_f1_winclip = max(f1_score(winclip_labels, (winclip_scores > t).astype(int), average='macro') for t in thresholds)
print("Zero-shot CLIP (ViT-B-16-plus-240) done!")
print(f"Zero-shot CLIP (ViT-B-16-plus-240) AUROC: {auroc_winclip:.4f}  Best F1: {best_f1_winclip:.4f}")

100%|██████████| 1621/1621 [02:31<00:00, 10.71it/s]


Zero-shot CLIP (ViT-B-16-plus-240) done!
Zero-shot CLIP (ViT-B-16-plus-240) AUROC: 0.5973  Best F1: 0.5280


**Zero-shot CLIP + FFT (ViT-B-16-plus-240)**

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

import open_clip
import torch.nn as nn
import sys
sys.path.append("/content/EECS545-Final-Project")
from FrequencyEncoder import FrequencyEncoder

# Load ViT-B-16-plus-240
model_b16, _, preprocess_b16 = open_clip.create_model_and_transforms(
    'ViT-B-16-plus-240', pretrained='laion400m_e32'
)
model_b16 = model_b16.to(device).eval()
tokenizer_b16 = open_clip.get_tokenizer('ViT-B-16-plus-240')

# FrequencyEncoder: out_dim=640 to match ViT-B-16-plus-240 embedding dim
freq_encoder_b16 = FrequencyEncoder(in_channels=3, out_dim=640).to(device)
freq_encoder_b16.eval()

# Build text features (611 abnormal, 104 normal)
normal_texts_b16 = [t.format(n.format(category)) for t in template_level_prompts for n in state_level_normal_prompts]
abnormal_texts_b16 = [t.format(a.format(category)) for t in template_level_prompts for a in state_level_abnormal_prompts]

with torch.no_grad():
    normal_tokens_b16 = tokenizer_b16(normal_texts_b16).to(device)
    abnormal_tokens_b16 = tokenizer_b16(abnormal_texts_b16).to(device)
    normal_features_b16 = model_b16.encode_text(normal_tokens_b16)
    abnormal_features_b16 = model_b16.encode_text(abnormal_tokens_b16)
    normal_features_b16 = normal_features_b16 / normal_features_b16.norm(dim=-1, keepdim=True)
    abnormal_features_b16 = abnormal_features_b16 / abnormal_features_b16.norm(dim=-1, keepdim=True)

# Mean pool normal into 1 vector, keep all 611 abnormal vectors
normal_feat_mean_b16 = normal_features_b16.mean(dim=0, keepdim=True)
normal_feat_mean_b16 = normal_feat_mean_b16 / normal_feat_mean_b16.norm(dim=-1, keepdim=True)

transform_b16 = transforms.Compose([
    transforms.Resize((240, 240), Image.BICUBIC),
    transforms.CenterCrop(240),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711])
])

scores_b16_fft, labels_b16_fft = [], []

with torch.no_grad():
    for _, row in tqdm(test_df_filtered.iterrows(), total=len(test_df_filtered)):
        img = Image.open(uid_to_frontal[row["uid"]]).convert("RGB")
        img_tensor = transform_b16(img).unsqueeze(0).to(device)

        # Spatial features from CLIP
        spatial_feat = model_b16.encode_image(img_tensor)  # (1, 640)
        spatial_feat = spatial_feat / spatial_feat.norm(dim=-1, keepdim=True)

        # Frequency features from FrequencyEncoder
        freq_feat = freq_encoder_b16(img_tensor.float())   # (1, 640)
        freq_feat = freq_feat / freq_feat.norm(dim=-1, keepdim=True)

        # Fuse: element-wise addition then re-normalize
        fused_feat = spatial_feat + freq_feat
        fused_feat = fused_feat / fused_feat.norm(dim=-1, keepdim=True)

        # Scoring: mean over 611 abnormal prompts minus normal
        sim_normal = (fused_feat @ normal_feat_mean_b16.T).item()
        sim_abnormal = (fused_feat @ abnormal_features_b16.T).mean().item()
        score = sim_abnormal - sim_normal

        scores_b16_fft.append(score)
        labels_b16_fft.append(row["abnormal"])

scores_b16_fft = np.array(scores_b16_fft)
labels_b16_fft = np.array(labels_b16_fft)

auroc_b16_fft = roc_auc_score(labels_b16_fft, scores_b16_fft)
thresholds = np.linspace(scores_b16_fft.min(), scores_b16_fft.max(), 100)
best_f1_b16_fft = max(f1_score(labels_b16_fft, (scores_b16_fft > t).astype(int), average='macro') for t in thresholds)
print(f"Zero-shot CLIP + FFT (ViT-B-16-plus-240) AUROC: {auroc_b16_fft:.4f}  Best F1: {best_f1_b16_fft:.4f}")

100%|██████████| 1621/1621 [02:36<00:00, 10.33it/s]


Zero-shot CLIP + FFT (ViT-B-16-plus-240) AUROC: 0.5807  Best F1: 0.5177


**WinCLIP (no FFT)**

In [ ]:
import sys
mods_to_delete = [key for key in sys.modules.keys()
                  if 'WinCLIP' in key or 'WinClip' in key or 'CLIPAD' in key]
for mod in mods_to_delete:
    del sys.modules[mod]

sys.path.insert(0, "/content/EECS545-Final-Project/WinClip")

from WinCLIP.model import WinClipAD

class WinClipAD_NoFFT(WinClipAD):
    @torch.no_grad()
    def encode_image(self, image):
        if self.precision == "fp16":
            image = image.half()
        image_features = self.model.encode_image(image)
        return [f / f.norm(dim=-1, keepdim=True) for f in image_features]

scales = [2, 3]
winclip_model_nofft = WinClipAD_NoFFT(
    out_size_h=224, out_size_w=224, device=device,
    backbone='ViT-B-16-plus-240', pretrained_dataset='laion400m_e32',
    scales=scales, img_resize=240, img_cropsize=240
)
winclip_model_nofft.build_text_feature_gallery("chest x-ray patient")

winclip_scores, winclip_labels = [], []
for _, row in tqdm(test_df_filtered.iterrows(), total=len(test_df_filtered)):
    img = Image.open(uid_to_frontal[row["uid"]]).convert("RGB")
    img_tensor = winclip_model_nofft.transform(img).unsqueeze(0).to(device)
    anomaly_maps = winclip_model_nofft(img_tensor)

    flat = anomaly_maps[0].ravel()
    topk = np.sort(flat)[-50:]
    winclip_scores.append(float(np.mean(topk)))
    winclip_labels.append(row["abnormal"])

winclip_scores = np.array(winclip_scores)
winclip_labels = np.array(winclip_labels)
auroc_winclip = roc_auc_score(winclip_labels, winclip_scores)
thresholds = np.linspace(winclip_scores.min(), winclip_scores.max(), 100)
best_f1_winclip = max(f1_score(winclip_labels, (winclip_scores > t).astype(int), average='macro') for t in thresholds)
print(f"WinCLIP (no FFT) AUROC: {auroc_winclip:.4f}  Best F1: {best_f1_winclip:.4f}")

self.grid_size (15, 15)
fusion version: textual_visual


100%|██████████| 1621/1621 [07:38<00:00,  3.54it/s]

WinCLIP (no FFT) AUROC: 0.5040  Best F1: 0.5031


**WinCLIP (no FFT, multi-prompt mean)**

In [ ]:
import sys
mods_to_delete = [key for key in sys.modules.keys()
    if 'WinCLIP' in key or 'WinClip' in key or 'CLIPAD' in key]
for mod in mods_to_delete:
    del sys.modules[mod]

sys.path.insert(0, "/content/EECS545-Final-Project/WinClip")
from WinCLIP.model import WinClipAD
from WinCLIP.ad_prompts import (
    template_level_prompts,
    state_level_normal_prompts,
    state_level_abnormal_prompts
)

class WinClipAD_NoFFT_MultiPrompt(WinClipAD):

    @torch.no_grad()
    def encode_image(self, image):
        # Skip FrequencyEncoder, use CLIP spatial features only
        if self.precision == "fp16":
            image = image.half()
        image_features = self.model.encode_image(image)
        return [f / f.norm(dim=-1, keepdim=True) for f in image_features]

    def build_text_feature_gallery(self, category: str):
        # Build all normal and abnormal phrases
        normal_phrases = [
            t.format(n.format(category))
            for t in template_level_prompts
            for n in state_level_normal_prompts
        ]
        abnormal_phrases = [
            t.format(a.format(category))
            for t in template_level_prompts
            for a in state_level_abnormal_prompts
        ]

        normal_tokens = self.tokenizer(normal_phrases).to(self.device)
        abnormal_tokens = self.tokenizer(abnormal_phrases).to(self.device)

        # Encode all prompts individually
        normal_features = self.encode_text(normal_tokens)    # (104, 640)
        abnormal_features = self.encode_text(abnormal_tokens)  # (611, 640)

        # Normal: mean into 1 vector (normal is homogeneous, mean is fine)
        avr_normal = normal_features.mean(dim=0, keepdim=True)
        avr_normal = avr_normal / avr_normal.norm(dim=-1, keepdim=True)

        # Abnormal: mean into 1 vector BUT we also store all vectors for scoring
        avr_abnormal = abnormal_features.mean(dim=0, keepdim=True)
        avr_abnormal = avr_abnormal / avr_abnormal.norm(dim=-1, keepdim=True)

        # Store all abnormal features for max/mean scoring in calculate_textual_anomaly_score
        abnormal_features = abnormal_features / abnormal_features.norm(dim=-1, keepdim=True)
        self.all_abnormal_features = abnormal_features  # (611, 640)

        self.avr_normal_text_features = avr_normal
        self.avr_abnormal_text_features = avr_abnormal
        self.text_features = torch.cat([avr_normal, avr_abnormal], dim=0)
        self.text_features /= self.text_features.norm(dim=-1, keepdim=True)

    @torch.no_grad()
    def calculate_textual_anomaly_score(self, visual_features):
        # Override: use mean over all 611 abnormal vectors instead of single averaged vector
        N = visual_features[0].shape[0]
        scale_anomaly_scores = []
        token_anomaly_scores = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))
        token_weights = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))

        for indx, (features, mask) in enumerate(zip(visual_features, self.masks)):
            # features: (N, 640)
            # sim_normal: (N,)
            sim_normal = (features.float() @ self.avr_normal_text_features.float().T).squeeze(-1)
            # sim_abnormal: mean over all 611 prompts → (N,)
            sim_abnormal = (features.float() @ self.all_abnormal_features.float().T).mean(dim=-1)

            # Softmax-style normality score
            sim = torch.stack([sim_normal, sim_abnormal], dim=-1) * 100.0
            normality_score = sim.softmax(dim=-1)[:, 0].cpu()

            mask = mask.reshape(-1)
            score = (1. / normality_score.clamp(min=1e-6)).unsqueeze(1).repeat(1, mask.shape[0])

            cur_token_anomaly_score = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))
            cur_token_anomaly_score[:, mask] += score
            cur_token_weight = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))
            cur_token_weight[:, mask] += 1.

            if indx in self.scale_begin_indx[1:]:
                token_weights = torch.clamp(token_weights, min=1e-6)
                token_anomaly_scores = token_anomaly_scores / token_weights
                scale_anomaly_scores.append(token_anomaly_scores)
                token_anomaly_scores = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))
                token_weights = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))

            token_weights += cur_token_weight
            token_anomaly_scores += cur_token_anomaly_score

        token_weights = torch.clamp(token_weights, min=1e-6)
        token_anomaly_scores = token_anomaly_scores / token_weights
        scale_anomaly_scores.append(token_anomaly_scores)
        scale_anomaly_scores = torch.stack(scale_anomaly_scores, dim=0)
        scale_anomaly_scores = torch.mean(scale_anomaly_scores, dim=0)
        scale_anomaly_scores = 1. - 1. / scale_anomaly_scores

        anomaly_map = scale_anomaly_scores.reshape(
            (N, self.grid_size[0], self.grid_size[1])
        ).unsqueeze(1)
        return anomaly_map


# Init model
scales = [2, 3]
winclip_model = WinClipAD_NoFFT_MultiPrompt(
    out_size_h=224, out_size_w=224, device=device,
    backbone='ViT-B-16-plus-240', pretrained_dataset='laion400m_e32',
    scales=scales, img_resize=240, img_cropsize=240
)
winclip_model.build_text_feature_gallery("patient")

# Run inference
winclip_scores, winclip_labels = [], []
for _, row in tqdm(test_df_filtered.iterrows(), total=len(test_df_filtered)):
    img = Image.open(uid_to_frontal[row["uid"]]).convert("RGB")
    img_tensor = winclip_model.transform(img).unsqueeze(0).to(device)
    anomaly_maps = winclip_model(img_tensor)

    # Top-50 mean for final image-level score
    flat = anomaly_maps[0].ravel()
    topk = np.sort(flat)[-50:]
    winclip_scores.append(float(np.mean(topk)))
    winclip_labels.append(row["abnormal"])

winclip_scores = np.array(winclip_scores)
winclip_labels = np.array(winclip_labels)

auroc = roc_auc_score(winclip_labels, winclip_scores)
thresholds = np.linspace(winclip_scores.min(), winclip_scores.max(), 100)
best_f1 = max(f1_score(winclip_labels, (winclip_scores > t).astype(int), average='macro') for t in thresholds)
print(f"WinCLIP (no FFT, multi-prompt mean) AUROC: {auroc:.4f}  Best F1: {best_f1:.4f}")

self.grid_size (15, 15)
fusion version: textual_visual


100%|██████████| 1621/1621 [08:55<00:00,  3.03it/s]

WinCLIP (no FFT, multi-prompt mean) AUROC: 0.5715  Best F1: 0.5279


**WinCLIP + FFT**

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

import sys

# clear cache
mods_to_delete = [key for key in sys.modules.keys()
                  if 'WinCLIP' in key or 'WinClip' in key or 'CLIPAD' in key]
for mod in mods_to_delete:
    del sys.modules[mod]

sys.path.insert(0, "/content/EECS545-Final-Project/WinClip")
from WinCLIP.model import WinClipAD

scales = [2, 3]
winclip_fft_model = WinClipAD(
    out_size_h=224, out_size_w=224, device=device,
    backbone='ViT-B-16-plus-240', pretrained_dataset='laion400m_e32',
    scales=scales, img_resize=240, img_cropsize=240
)
winclip_fft_model.build_text_feature_gallery("chest x-ray patient")

scores_wfft, labels_wfft = [], []
for _, row in tqdm(test_df_filtered.iterrows(), total=len(test_df_filtered)):
    img = Image.open(uid_to_frontal[row["uid"]]).convert("RGB")
    img_tensor = winclip_fft_model.transform(img).unsqueeze(0).to(device)
    anomaly_maps = winclip_fft_model(img_tensor)

    flat = anomaly_maps[0].ravel()
    topk = np.sort(flat)[-50:]
    scores_wfft.append(float(np.mean(topk)))
    labels_wfft.append(row["abnormal"])

scores_wfft = np.array(scores_wfft)
labels_wfft = np.array(labels_wfft)
auroc_wfft = roc_auc_score(labels_wfft, scores_wfft)
thresholds = np.linspace(scores_wfft.min(), scores_wfft.max(), 100)
best_f1_wfft = max(f1_score(labels_wfft, (scores_wfft > t).astype(int), average='macro') for t in thresholds)
print(f"WinCLIP + FFT AUROC: {auroc_wfft:.4f}  Best F1: {best_f1_wfft:.4f}")

self.grid_size (15, 15)
fusion version: textual_visual


100%|██████████| 1621/1621 [07:48<00:00,  3.46it/s]

WinCLIP + FFT AUROC: 0.5034  Best F1: 0.5052


**WinCLIP + FFT + multi-prompt mean**

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

import sys
mods_to_delete = [key for key in sys.modules.keys()
    if 'WinCLIP' in key or 'WinClip' in key or 'CLIPAD' in key]
for mod in mods_to_delete:
    del sys.modules[mod]

sys.path.insert(0, "/content/EECS545-Final-Project/WinClip")
from WinCLIP.model import WinClipAD
from WinCLIP.ad_prompts import (
    template_level_prompts,
    state_level_normal_prompts,
    state_level_abnormal_prompts
)

class WinClipAD_FFT_MultiPrompt(WinClipAD):
    # Keep original encode_image (with FFT fusion)

    def build_text_feature_gallery(self, category: str):
        normal_phrases = [
            t.format(n.format(category))
            for t in template_level_prompts
            for n in state_level_normal_prompts
        ]
        abnormal_phrases = [
            t.format(a.format(category))
            for t in template_level_prompts
            for a in state_level_abnormal_prompts
        ]

        normal_tokens = self.tokenizer(normal_phrases).to(self.device)
        abnormal_tokens = self.tokenizer(abnormal_phrases).to(self.device)

        normal_features = self.encode_text(normal_tokens)
        abnormal_features = self.encode_text(abnormal_tokens)

        avr_normal = normal_features.mean(dim=0, keepdim=True)
        avr_normal = avr_normal / avr_normal.norm(dim=-1, keepdim=True)

        avr_abnormal = abnormal_features.mean(dim=0, keepdim=True)
        avr_abnormal = avr_abnormal / avr_abnormal.norm(dim=-1, keepdim=True)

        abnormal_features = abnormal_features / abnormal_features.norm(dim=-1, keepdim=True)
        self.all_abnormal_features = abnormal_features

        self.avr_normal_text_features = avr_normal
        self.avr_abnormal_text_features = avr_abnormal
        self.text_features = torch.cat([avr_normal, avr_abnormal], dim=0)
        self.text_features /= self.text_features.norm(dim=-1, keepdim=True)

    @torch.no_grad()
    def calculate_textual_anomaly_score(self, visual_features):
        N = visual_features[0].shape[0]
        scale_anomaly_scores = []
        token_anomaly_scores = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))
        token_weights = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))

        for indx, (features, mask) in enumerate(zip(visual_features, self.masks)):
            sim_normal = (features.float() @ self.avr_normal_text_features.float().T).squeeze(-1)
            sim_abnormal = (features.float() @ self.all_abnormal_features.float().T).mean(dim=-1)

            sim = torch.stack([sim_normal, sim_abnormal], dim=-1) * 100.0
            normality_score = sim.softmax(dim=-1)[:, 0].cpu()

            mask = mask.reshape(-1)
            score = (1. / normality_score.clamp(min=1e-6)).unsqueeze(1).repeat(1, mask.shape[0])

            cur_token_anomaly_score = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))
            cur_token_anomaly_score[:, mask] += score
            cur_token_weight = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))
            cur_token_weight[:, mask] += 1.

            if indx in self.scale_begin_indx[1:]:
                token_weights = torch.clamp(token_weights, min=1e-6)
                token_anomaly_scores = token_anomaly_scores / token_weights
                scale_anomaly_scores.append(token_anomaly_scores)
                token_anomaly_scores = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))
                token_weights = torch.zeros((N, self.grid_size[0] * self.grid_size[1]))

            token_weights += cur_token_weight
            token_anomaly_scores += cur_token_anomaly_score

        token_weights = torch.clamp(token_weights, min=1e-6)
        token_anomaly_scores = token_anomaly_scores / token_weights
        scale_anomaly_scores.append(token_anomaly_scores)
        scale_anomaly_scores = torch.stack(scale_anomaly_scores, dim=0)
        scale_anomaly_scores = torch.mean(scale_anomaly_scores, dim=0)
        scale_anomaly_scores = 1. - 1. / scale_anomaly_scores

        anomaly_map = scale_anomaly_scores.reshape(
            (N, self.grid_size[0], self.grid_size[1])
        ).unsqueeze(1)
        return anomaly_map


# Init model
scales = [2, 3]
winclip_fft_model = WinClipAD_FFT_MultiPrompt(
    out_size_h=224, out_size_w=224, device=device,
    backbone='ViT-B-16-plus-240', pretrained_dataset='laion400m_e32',
    scales=scales, img_resize=240, img_cropsize=240
)
winclip_fft_model.build_text_feature_gallery("patient")

# Run inference
winclip_scores, winclip_labels = [], []
for _, row in tqdm(test_df_filtered.iterrows(), total=len(test_df_filtered)):
    img = Image.open(uid_to_frontal[row["uid"]]).convert("RGB")
    img_tensor = winclip_fft_model.transform(img).unsqueeze(0).to(device)
    anomaly_maps = winclip_fft_model(img_tensor)

    flat = anomaly_maps[0].ravel()
    topk = np.sort(flat)[-50:]
    winclip_scores.append(float(np.mean(topk)))
    winclip_labels.append(row["abnormal"])

winclip_scores = np.array(winclip_scores)
winclip_labels = np.array(winclip_labels)

auroc = roc_auc_score(winclip_labels, winclip_scores)
thresholds = np.linspace(winclip_scores.min(), winclip_scores.max(), 100)
best_f1 = max(f1_score(winclip_labels, (winclip_scores > t).astype(int), average='macro') for t in thresholds)
print(f"WinCLIP + FFT (multi-prompt mean) AUROC: {auroc:.4f}  Best F1: {best_f1:.4f}")

self.grid_size (15, 15)
fusion version: textual_visual


100%|██████████| 1621/1621 [09:02<00:00,  2.99it/s]

WinCLIP + FFT (multi-prompt mean) AUROC: 0.5715  Best F1: 0.5294


In [ ]:
# from sklearn.metrics import roc_auc_score, precision_recall_curve, f1_score

# def evaluate(scores, labels, model_name):
#     auroc = roc_auc_score(labels, scores)
#     thresholds = np.linspace(scores.min(), scores.max(), 100)
#     best_f1 = max(f1_score(labels, (scores > t).astype(int), average='macro') for t in thresholds)
#     print(f"{model_name:<45} {auroc:>8.4f} {best_f1:>10.4f}")

# print(f"{'Model':<45} {'AUROC':>8} {'Macro F1':>10}")
# print("-" * 65)
# evaluate(scores_clip,         labels_clip,         "Zero-shot CLIP (ViT-B-32)")
# evaluate(scores_fft,          labels_fft,           "Zero-shot CLIP + FFT (ViT-B-32)")
# evaluate(scores_b16,          labels_b16,           "Zero-shot CLIP (ViT-B-16-plus-240)")
# evaluate(scores_b16_fft,      labels_b16_fft,       "Zero-shot CLIP + FFT (ViT-B-16-plus-240)")
# evaluate(winclip_scores_nofft, winclip_labels_nofft, "WinCLIP (no FFT)")
# evaluate(winclip_scores_fft,   winclip_labels_fft,   "WinCLIP + FFT")
# evaluate(winclip_scores_mp,    winclip_labels_mp,    "WinCLIP (no FFT, multi-prompt mean)")
# evaluate(winclip_scores_mpfft, winclip_labels_mpfft, "WinCLIP + FFT (multi-prompt mean)")

Model                                  AUROC  Best F1
----------------------------------------------------------
Zero-shot CLIP                      AUROC: 0.5962  Best F1: 0.9229
Zero-shot CLIP + FFT                AUROC: 0.6037  Best F1: 0.9229
WinCLIP                             AUROC: 0.5715  Best F1: 0.9229
WinCLIP + FFT                       AUROC: 0.5034  Best F1: 0.9229


In [ ]:
# @title
import matplotlib.pyplot as plt

# (1) Score distribution
plt.figure(figsize=(12, 4))
plt.hist(scores_clip,    bins=50, alpha=0.5, label="Zero-shot CLIP")
plt.hist(scores_fft,     bins=50, alpha=0.5, label="Zero-shot CLIP + FFT")
plt.hist(winclip_scores, bins=50, alpha=0.5, label="WinCLIP")
plt.hist(scores_wfft,    bins=50, alpha=0.5, label="WinCLIP + FFT")
plt.xlabel("Anomaly Score")
plt.ylabel("Count")
plt.title("Score Distribution")
plt.legend()
plt.show()

# (2) Class-wise separation
for scores, labels, name in [
    (scores_clip,    labels_clip,    "Zero-shot CLIP"),
    (scores_fft,     labels_fft,     "Zero-shot CLIP + FFT"),
    (winclip_scores, winclip_labels, "WinCLIP"),
    (scores_wfft,    labels_wfft,    "WinCLIP + FFT"),
]:
    normal_mean   = scores[labels == 0].mean()
    abnormal_mean = scores[labels == 1].mean()
    diff = abnormal_mean - normal_mean
    print(f"{name}")
    print(f"  Normal mean:   {normal_mean:.4f}")
    print(f"  Abnormal mean: {abnormal_mean:.4f}")
    print(f"  Difference:    {diff:.4f}")
    print()

# Previous Test

In [ ]:
# @title
import zipfile
import os

zip_path = "/content/drive/MyDrive/chest-xrays-indiana-university.zip"
extract_path = "/content/images"

if not os.path.exists(extract_path):
    print("Extracting images, please wait...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Done!")
else:
    print("Already extracted!")

# 確認解壓縮結果
!ls /content/images
# @title
# 確認所有路徑都存在
import os

paths = {
    "images": "/content/images/chest-xrays-indiana-university/images/valid_uid_images",
    "csv": "/content/images/chest-xrays-indiana-university/final_clean_dataset.csv",
    "projections": "/content/images/chest-xrays-indiana-university/indiana_projections.csv",
    "WinCLIP": "/content/EECS545-Final-Project/WinClip",
    "FrequencyEncoder": "/content/EECS545-Final-Project/FrequencyEncoder.py"
}

for name, path in paths.items():
    print(f"{name}: {'✅ exists' if os.path.exists(path) else '❌ NOT FOUND'}")
# @title
import sys
import os
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score

# Add paths
sys.path.append("/content/EECS545-Final-Project")
sys.path.append("/content/EECS545-Final-Project/WinClip")

# Paths
IMAGE_FOLDER = "/content/images/chest-xrays-indiana-university/images/valid_uid_images"
CSV_PATH = "/content/images/chest-xrays-indiana-university/final_clean_dataset.csv"
PROJ_PATH = "/content/images/chest-xrays-indiana-university/indiana_projections.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ===============================
# Load data
# ===============================
df_text = pd.read_csv(CSV_PATH)
df_text["uid"] = df_text["uid"].astype(str)
df_text = df_text[["uid", "normal"]].dropna()
df_text["abnormal"] = 1 - df_text["normal"].astype(int)

proj_df = pd.read_csv(PROJ_PATH)
proj_df["uid"] = proj_df["uid"].astype(str)
proj_df["projection"] = proj_df["projection"].str.strip().str.lower()

# Build uid -> frontal image path
uid_to_frontal = {}
for uid, group in proj_df.groupby("uid"):
    frontal = group[group["projection"] == "frontal"]
    if len(frontal) == 1:
        fname = frontal.iloc[0]["filename"]
        fpath = os.path.join(IMAGE_FOLDER, fname)
        if os.path.exists(fpath):
            uid_to_frontal[uid] = fpath

# Match with labels
df = df_text[df_text["uid"].isin(uid_to_frontal.keys())].copy()
df = df.drop_duplicates(subset=["uid"]).reset_index(drop=True)
print(f"Total samples: {len(df)}")
print(f"Abnormal: {df['abnormal'].sum()}, Normal: {(df['abnormal']==0).sum()}")

**Simplified WinCLIP**
Backbone: ViT-B-32

In [ ]:
# @title
import open_clip
from torchvision import transforms

# Load CLIP model
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32',
    pretrained='openai'
)
model = model.to(device)
model.eval()
tokenizer = open_clip.get_tokenizer('ViT-B-32')
print("CLIP model loaded!")
# @title
# Medical text prompts
normal_texts = [
    "a chest x-ray without abnormality",
    "a normal chest x-ray",
    "a healthy chest x-ray without findings",
    "a chest x-ray within normal limits",
    "a chest x-ray with clear lung fields",
]

abnormal_texts = [
    "a chest x-ray with pneumothorax",
    "a chest x-ray with pleural effusion",
    "a chest x-ray with cardiomegaly",
    "a chest x-ray with abnormal findings",
    "a chest x-ray with pathology",
    "a chest x-ray with pulmonary edema",
    "a chest x-ray with consolidation",
]

with torch.no_grad():
    normal_tokens = tokenizer(normal_texts).to(device)
    abnormal_tokens = tokenizer(abnormal_texts).to(device)
    normal_features = model.encode_text(normal_tokens)
    abnormal_features = model.encode_text(abnormal_tokens)
    normal_features = normal_features.mean(dim=0, keepdim=True)
    abnormal_features = abnormal_features.mean(dim=0, keepdim=True)
    normal_features = normal_features / normal_features.norm(dim=-1, keepdim=True)
    abnormal_features = abnormal_features / abnormal_features.norm(dim=-1, keepdim=True)

print("Text features ready!")
# @title
from PIL import ImageOps

def resize_and_pad(img, target_size=(224, 224)):
    img = ImageOps.contain(img, target_size)
    canvas = Image.new("RGB", target_size, color=(0, 0, 0))
    paste_x = (target_size[0] - img.size[0]) // 2
    paste_y = (target_size[1] - img.size[1]) // 2
    canvas.paste(img, (paste_x, paste_y))
    return canvas

all_scores = []
all_labels = []

with torch.no_grad():
    for _, row in tqdm(df.iterrows(), total=len(df)):
        uid = row["uid"]
        label = row["abnormal"]
        img_path = uid_to_frontal[uid]

        # Load and preprocess image
        img = Image.open(img_path).convert("RGB")
        img = resize_and_pad(img)
        img_tensor = preprocess(img).unsqueeze(0).to(device)

        # Get image features
        image_features = model.encode_image(img_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Compute similarity scores
        normal_score = (image_features @ normal_features.T).item()
        abnormal_score = (image_features @ abnormal_features.T).item()

        # Anomaly score = abnormal - normal
        anomaly_score = abnormal_score - normal_score

        all_scores.append(anomaly_score)
        all_labels.append(label)

print("Inference done!")
# @title
all_scores = np.array(all_scores)
all_labels = np.array(all_labels)

# AUROC
auroc = roc_auc_score(all_labels, all_scores)

# F1 (threshold = 0)
preds = (all_scores > 0).astype(int)
f1 = f1_score(all_labels, preds)

print(f"WinCLIP (Zero-shot CLIP) Results:")
print(f"AUROC:    {auroc:.4f}")
print(f"F1 Score: {f1:.4f}")

**With FrequencyEncoder**

In [ ]:
# @title
sys.path.append("/content/EECS545-Final-Project")
from FrequencyEncoder import FrequencyEncoder

# Load FrequencyEncoder
freq_encoder = FrequencyEncoder(in_channels=3, out_dim=512).to(device)
freq_encoder.eval()
print("FrequencyEncoder loaded!")

# Re-run inference with FrequencyEncoder
from torchvision import transforms

to_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711])
])

all_scores_fft = []
all_labels_fft = []

with torch.no_grad():
    for _, row in tqdm(df.iterrows(), total=len(df)):
        uid = row["uid"]
        label = row["abnormal"]
        img_path = uid_to_frontal[uid]

        img = Image.open(img_path).convert("RGB")
        img = resize_and_pad(img)

        # CLIP features
        img_clip = preprocess(img).unsqueeze(0).to(device)
        clip_features = model.encode_image(img_clip)
        clip_features = clip_features / clip_features.norm(dim=-1, keepdim=True)

        # Frequency features
        img_tensor = to_tensor(img).unsqueeze(0).to(device)
        freq_features = freq_encoder(img_tensor)
        freq_features = freq_features / freq_features.norm(dim=-1, keepdim=True)

        # Fuse
        fused = clip_features + freq_features
        fused = fused / fused.norm(dim=-1, keepdim=True)

        # Anomaly score
        normal_score = (fused @ normal_features.T).item()
        abnormal_score = (fused @ abnormal_features.T).item()
        anomaly_score = abnormal_score - normal_score

        all_scores_fft.append(anomaly_score)
        all_labels_fft.append(label)

all_scores_fft = np.array(all_scores_fft)
all_labels_fft = np.array(all_labels_fft)

auroc_fft = roc_auc_score(all_labels_fft, all_scores_fft)
preds_fft = (all_scores_fft > 0).astype(int)
f1_fft = f1_score(all_labels_fft, preds_fft)

print(f"\nResults Comparison:")
print(f"{'Model':<30} {'AUROC':>8} {'F1':>8}")
print(f"{'Zero-shot CLIP':<30} {auroc:.4f} {f1:.4f}")
print(f"{'Zero-shot CLIP + FFT':<30} {auroc_fft:.4f} {f1_fft:.4f}")

**Full WinCLIP**

In [ ]:
# @title
!git pull origin yslin
# @title
!pip install -q open_clip_torch
import open_clip

# 確認這個 backbone 存在
models = open_clip.list_pretrained()
vit_models = [(m, p) for m, p in models if 'ViT-B-16-plus' in m]
print(vit_models)
# @title
import open_clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading ViT-B-16-plus-240...")
model_winclip, _, preprocess_winclip = open_clip.create_model_and_transforms(
    'ViT-B-16-plus-240',
    pretrained='laion400m_e32'
)
model_winclip = model_winclip.to(device)
model_winclip.eval()
tokenizer_winclip = open_clip.get_tokenizer('ViT-B-16-plus-240')

print("Model loaded!")
print(f"Device: {device}")
# @title
import subprocess
result = subprocess.run(['git', 'pull', 'origin', 'yslin'],
                      capture_output=True, text=True,
                      cwd='/content/EECS545-Final-Project')
print(result.stdout)
print(result.stderr)
# @title
import os

# 確認完整結構
print("WinClip 裡面有：")
print(os.listdir("/content/EECS545-Final-Project/WinClip"))

print("\nWinCLIP 裡面有：")
print(os.listdir("/content/EECS545-Final-Project/WinClip/WinCLIP"))
# @title
import sys

# 確保路徑正確
if "/content/EECS545-Final-Project/WinClip" not in sys.path:
    sys.path.insert(0, "/content/EECS545-Final-Project/WinClip")

print("目前 sys.path 前三個：")
print(sys.path[:3])

from WinCLIP.model import WinClipAD
print("成功！")
# @title
import sys
sys.path.append("/content/EECS545-Final-Project/WinClip")

from WinCLIP.model import WinClipAD
print("WinClipAD imported successfully!")
# @title
!git pull origin yslin
# @title
# Initialize WinClipAD
scales = [2, 3]  # multi-scale windows

winclip_model = WinClipAD(
    out_size_h=224,
    out_size_w=224,
    device=device,
    backbone='ViT-B-16-plus-240',
    pretrained_dataset='laion400m_e32',
    scales=scales,
    img_resize=240,
    img_cropsize=240
)

print("WinClipAD initialized!")
print(f"Grid size: {winclip_model.grid_size}")
# @title
# Build text feature gallery with medical prompts
winclip_model.build_text_feature_gallery("chest x-ray patient")

print("Text feature gallery built!")
print(f"Text features shape: {winclip_model.text_features.shape}")
# @title
# 清除舊的 model
del winclip_model
import torch
torch.cuda.empty_cache()

# 重新初始化
import sys
sys.path.insert(0, "/content/EECS545-Final-Project/WinClip")

# 清除 module cache
mods_to_delete = [key for key in sys.modules.keys()
                  if 'WinCLIP' in key or 'WinClip' in key or 'CLIPAD' in key]
for mod in mods_to_delete:
    del sys.modules[mod]

from WinCLIP.model import WinClipAD

scales = [2, 3]
winclip_model = WinClipAD(
    out_size_h=224,
    out_size_w=224,
    device=device,
    backbone='ViT-B-16-plus-240',
    pretrained_dataset='laion400m_e32',
    scales=scales,
    img_resize=240,
    img_cropsize=240
)
winclip_model.build_text_feature_gallery("chest x-ray patient")
print("Done!")
# @title
for i in range(3):
    print(f"mask[{i}]: {winclip_model.masks[i]}")
    print(f"mask[{i}] shape: {winclip_model.masks[i].shape}")
    print()
# @title
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score

all_scores = []
all_labels = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    uid = row["uid"]
    label = row["abnormal"]
    img_path = uid_to_frontal[uid]

    # Load and preprocess image
    img = Image.open(img_path).convert("RGB")
    img_tensor = winclip_model.transform(img).unsqueeze(0).to(device)

    # Get anomaly map
    anomaly_maps = winclip_model(img_tensor)

    # Use mean of anomaly map as anomaly score
    score = float(np.mean(anomaly_maps[0]))
    all_scores.append(score)
    all_labels.append(label)

all_scores = np.array(all_scores)
all_labels = np.array(all_labels)

auroc = roc_auc_score(all_labels, all_scores)
preds = (all_scores > np.median(all_scores)).astype(int)
f1 = f1_score(all_labels, preds)

print(f"\nFull WinCLIP Results:")
print(f"AUROC:    {auroc:.4f}")
print(f"F1 Score: {f1:.4f}")
# @title
# 重新計算，用 max 而不是 mean
all_scores_max = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    uid = row["uid"]
    label = row["abnormal"]
    img_path = uid_to_frontal[uid]

    img = Image.open(img_path).convert("RGB")
    img_tensor = winclip_model.transform(img).unsqueeze(0).to(device)

    anomaly_maps = winclip_model(img_tensor)
    score = float(np.max(anomaly_maps[0]))  # max instead of mean
    all_scores_max.append(score)

all_scores_max = np.array(all_scores_max)
auroc_max = roc_auc_score(all_labels, all_scores_max)
preds_max = (all_scores_max > np.median(all_scores_max)).astype(int)
f1_max = f1_score(all_labels, preds_max)

print(f"WinCLIP (max pooling):")
print(f"AUROC:    {auroc_max:.4f}")
print(f"F1 Score: {f1_max:.4f}")
# @title
# Save current results before re-running
results_v1 = {
    "zero_shot_clip": {"auroc": auroc, "f1": f1},
    "zero_shot_clip_fft": {"auroc": auroc_fft, "f1": f1_fft},
    "winclip_mean": {"auroc": 0.4709, "f1": 0.5460},
    "winclip_max": {"auroc": 0.4726, "f1": 0.5455},
}

print("=== Results V1 (saved) ===")
for model_name, scores in results_v1.items():
    print(f"{model_name:<25} AUROC: {scores['auroc']:.4f}  F1: {scores['f1']:.4f}")
# @title
# Pull latest changes
!git pull origin yslin

# Rebuild text features with new prompts
winclip_model.build_text_feature_gallery("chest x-ray patient")
print("Text features rebuilt!")
# @title
# Re-run WinCLIP with updated prompts
all_scores_v2 = []
all_labels_v2 = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    uid = row["uid"]
    label = row["abnormal"]
    img_path = uid_to_frontal[uid]

    img = Image.open(img_path).convert("RGB")
    img_tensor = winclip_model.transform(img).unsqueeze(0).to(device)
    anomaly_maps = winclip_model(img_tensor)
    score = float(np.mean(anomaly_maps[0]))
    all_scores_v2.append(score)
    all_labels_v2.append(label)

all_scores_v2 = np.array(all_scores_v2)
all_labels_v2 = np.array(all_labels_v2)

auroc_v2 = roc_auc_score(all_labels_v2, all_scores_v2)
preds_v2 = (all_scores_v2 > np.median(all_scores_v2)).astype(int)
f1_v2 = f1_score(all_labels_v2, preds_v2)

print(f"\n=== Results Comparison ===")
print(f"{'Model':<25} {'AUROC':>8} {'F1':>8}")
print(f"{'WinCLIP V1 (mean)':<25} {0.4709:.4f} {0.5460:.4f}")
print(f"{'WinCLIP V2 (updated)':<25} {auroc_v2:.4f} {f1_v2:.4f}")
# @title
all_scores_winclip = []
all_labels_winclip = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    uid = row["uid"]
    label = row["abnormal"]
    img_path = uid_to_frontal[uid]

    img = Image.open(img_path).convert("RGB")
    img_tensor = winclip_model.transform(img).unsqueeze(0).to(device)
    anomaly_maps = winclip_model(img_tensor)

    score = float(np.max(anomaly_maps[0]))  # max 取代 mean
    all_scores_winclip.append(score)
    all_labels_winclip.append(label)

all_scores_winclip = np.array(all_scores_winclip)
all_labels_winclip = np.array(all_labels_winclip)
print("Done!")
# @title
from sklearn.metrics import roc_auc_score, precision_recall_curve

def evaluate(scores, labels, model_name):
    auroc = roc_auc_score(labels, scores)
    precision, recall, _ = precision_recall_curve(labels, scores)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    best_f1 = np.max(f1_scores)
    print(f"{model_name:<30} AUROC: {auroc:.4f}  Best F1: {best_f1:.4f}")

evaluate(all_scores_winclip, all_labels_winclip, "WinCLIP + FFT (max)")
# @title
# All models
print(f"{'Model':<35} {'AUROC':>8} {'Best F1':>8}")
print("-" * 55)
evaluate(all_scores,      all_labels,      "Zero-shot CLIP")
evaluate(all_scores_fft,  all_labels_fft,  "Zero-shot CLIP + FFT")
evaluate(all_scores_winclip, all_labels_winclip, "WinCLIP + FFT (max)")